---
title: Sharing compute artifacts
---

Once you have built a phasic graph and let phasic run the symbolic elimination once (e.g. by calling `graph.expectation()`), the result is cached on disk in `~/.phasic_cache/parameterized_reward_compute/<hash>.bin`. These binary files are the artifact that the `phasic.compute_repository` module shares.

The disk layout is **content-addressed** — file names come from `ptd_graph_content_hash`, which depends only on the graph's topology + coefficients (not on theta or the machine). That means a `.bin` produced on one machine plugs straight into the on-disk cache of another machine that defines a structurally identical graph.

The sharing system follows the same content-addressed model as IPFS and uses **progressive enhancement**:

| Tier | Configuration | How artifacts are fetched |
|------|---------------|---------------------------|
| 1 — Zero config | Nothing installed | Public HTTP gateways (`ipfs.io`, `dweb.link`, …) or `raw.githubusercontent.com` |
| 2 — Daemon installed | `ipfs daemon` running | Fetched peer-to-peer via the local kubo daemon; you can also pin to mirror |
| 3 — Publish | `gh` + daemon | Use `phasic publish-compute` to open a PR against the registry repo |

This tutorial walks through the consumer side (browse + fetch + pin), the publisher side (CLI), and how to plug in alternative transports (S3, GCS, etc.).

## Setup

In [1]:
import json, tempfile, gzip, shutil
from pathlib import Path

import phasic
from phasic.compute_repository import (
    ComputeRegistry,
    fetch_compute,
    compute_source,
    pin_compute,
    list_computes,
)
from phasic.transport import TransportBackend
from phasic.transport.ipfs import IPFSBackend
from phasic.exceptions import PTDBackendError, PTDFormatError

## What is a compute artifact, exactly?

Every published artifact is a single binary file produced by the C side via `ptd_save_parameterized_reward_compute_graph`. Its header starts with the magic bytes `b'PTDPRMC1'` followed by a 32-bit format revision, the truncated graph hash, and the serialised command list.

Two artifact flavours live in the registry:

- **parent** — the monolithic compute graph for the entire model. Written by Stage A2 of the C path.
- **scc_<hash>** — per-SCC compute graphs, written when the hierarchical composer is engaged. The C path can rebuild missing SCCs on demand, so per-SCC artifacts are optional, but bundling them speeds up the first call.

Both flavours sit in `~/.phasic_cache/parameterized_reward_compute/` alongside the local cache the C path populates.

## Inspecting backend status

Before sharing or fetching anything it helps to know what state the local IPFS environment is in:

In [2]:
backend = IPFSBackend()
status = backend.status()

print('Backend')
print('=' * 40)
print(f"name:           {status['name']}")
print(f"daemon up:      {status['daemon']}")
print(f"daemon version: {status['daemon_version'] or 'N/A'}")
print(f"gateways:       {len(status['gateways'])} configured")
for gw in status['gateways']:
    print(f"  - {gw}")

Backend
name:           ipfs
daemon up:      True
daemon version: 0.38.1
gateways:       4 configured
  - https://ipfs.io
  - https://cloudflare-ipfs.com
  - https://dweb.link
  - https://gateway.pinata.cloud


If `daemon up` is `False`, fetches fall back to public HTTP gateways automatically. Publishing and pinning require a daemon:

```bash
ipfs daemon &
```

## Demo registry

Real consumers point `ComputeRegistry` at `munch-group/phasic-traces`. For this tutorial we set up a local registry and a fake transport backend so every cell executes end-to-end even without internet access.

In [3]:
demo_root = Path(tempfile.mkdtemp(prefix='phasic_share_demo_'))
registry_dir = demo_root / 'registry_cache'
fake_origin  = demo_root / 'fake_origin'
fake_origin.mkdir(parents=True)

# We'll build a small parameterised graph, run elimination once
# locally so the .bin lands on disk, then 'publish' it by copying
# the file into our fake origin and writing a registry.json that
# references it.

g = phasic.Graph(1)
v0 = g.starting_vertex()
v1 = g.find_or_create_vertex([1])
v2 = g.find_or_create_vertex([2])
v0.add_edge(v1, [1.0])  # parameterised: weight = c * theta[0]
v1.add_edge(v2, [1.0])
g.update_weights([2.0])
g.expectation()  # populates the C-side compute graph

hash_hex = phasic.hash.compute_graph_hash(g).hash_hex
print(f'graph hash: {hash_hex[:16]}…')
print(f'graph hash full: {hash_hex}')

graph hash: 29dc136099b6cf8d…
graph hash full: 29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c


In [4]:
# Save the .bin to our fake origin (simulating the publish step).
import hashlib

publish_path = fake_origin / f'{hash_hex}.bin'
g._save_param_compute_graph(str(publish_path))
size = publish_path.stat().st_size
sha = hashlib.sha256(publish_path.read_bytes()).hexdigest()
print(f'wrote {size:,} bytes to {publish_path.name}')
print(f'sha256: {sha[:32]}…')

wrote 2,304 bytes to 29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c.bin
sha256: 94be619b8995158b1ed4e5e496a8d18e…


In [5]:
# Build a registry.json pointing at the fake artifact.
registry_dir.mkdir(parents=True)
registry_data = {
    'version': '2.0',
    'format':  'ptd_pcg',
    'computes': {
        'demo_chain_2step': {
            'graph_hash':      hash_hex,
            'format_revision': 2,
            'artifacts': {
                'parent': {
                    # In a real registry this is a github-relative
                    # path or an IPFS CID. We use a local file:// URL
                    # so the gateway client can fetch it.
                    'cid_or_path': f'file://{publish_path}',
                    'sha256':      sha,
                    'size_bytes':  size,
                },
                'scc': [],
            },
            'metadata': {
                'description': 'Two-step parameterised chain (demo)',
                'domain':      'demo',
                'model_type':  'chain',
                'vertices':    g.vertices_length(),
                'param_length': 1,
                'author':      'demo',
                'license':     'MIT',
                'tags':        ['demo', 'tutorial'],
            },
        },
    },
}
(registry_dir / 'registry.json').write_text(
    json.dumps(registry_data, indent=2))
print('demo registry seeded')

demo registry seeded


### A fake transport backend for file:// URLs

The default `IPFSBackend` knows about CIDs and HTTPS gateways. To keep this tutorial self-contained we plug in a tiny `TransportBackend` subclass that fetches from `file://` URLs.

In [6]:
from urllib.parse import urlparse

class LocalFileBackend(TransportBackend):
    @property
    def name(self) -> str:
        return 'local'

    def get(self, cid_or_url, output_path):
        parsed = urlparse(cid_or_url)
        if parsed.scheme != 'file':
            raise PTDBackendError(
                f'LocalFileBackend only handles file:// URLs, '
                f'got {cid_or_url!r}')
        shutil.copy2(parsed.path, output_path)

    def add(self, path):
        raise PTDBackendError('LocalFileBackend cannot publish')

## Listing what's available

`list_computes()` returns one dict per published artifact. Filtering by `domain`, `model_type`, or `tags` narrows the list.

In [7]:
registry = ComputeRegistry(
    registry_repo='demo/demo',
    cache_dir=registry_dir,
    backend=LocalFileBackend(),
    auto_update=False)  # use the seeded registry.json

for entry in registry.list_computes():
    print(f"{entry['compute_id']:25s}  "
          f"{entry.get('vertices', '?')} vertices, "
          f"hash {entry['graph_hash'][:16]}…")

demo_chain_2step           3 vertices, hash 29dc136099b6cf8d…


In [8]:
# Filter by tags
tutorial_entries = registry.list_computes(tags=['tutorial'])
print(f'{len(tutorial_entries)} entries tagged "tutorial"')

1 entries tagged "tutorial"


## Inspecting where a fetch would come from

`compute_source(graph)` reports the cheapest source for *graph*'s artifact without actually downloading anything:

In [9]:
# Build the same graph in a fresh Python session would normally
# come up with no cached file. We move our local file aside to
# simulate that here.
import os
from phasic.compute_repository import _param_compute_cache_dir

cache_dir = _param_compute_cache_dir()
local_bin = cache_dir / f'{hash_hex}.bin'
moved_aside = None
if local_bin.exists():
    moved_aside = local_bin.with_suffix('.bin.tutorial_backup')
    local_bin.rename(moved_aside)

info = registry.compute_source(g)
for k, v in info.items():
    print(f'  {k:18s}: {v}')

if moved_aside is not None:
    moved_aside.rename(local_bin)

  graph_hash        : 29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c
  local_path        : /Users/kmt/.phasic_cache/parameterized_reward_compute/29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c.bin
  cached_locally    : False
  registered        : True
  compute_id        : demo_chain_2step
  backend           : local
  source            : local


## Fetching an artifact

`fetch_compute(graph)` looks the graph up by content hash, downloads the artifact via the transport backend, verifies the SHA-256, and writes it atomically into the local cache. Returns `True` on a hit, `False` if the registry has no entry for the graph's hash.

In [10]:
# Start from a clean slate.
if local_bin.exists():
    local_bin.unlink()

hit = registry.fetch_compute(g)
print(f'cache hit: {hit}')
print(f'file present: {local_bin.exists()}')
print(f'size: {local_bin.stat().st_size:,} bytes')

cache hit: True
file present: True
size: 2,304 bytes


The next `g.expectation()` call now finds the file via the normal C-side lookup — no Python plumbing required:

In [11]:
# Wipe the in-memory compute graph so we exercise the on-disk path.
g2 = phasic.Graph(1)
u0 = g2.starting_vertex()
u1 = g2.find_or_create_vertex([1])
u2 = g2.find_or_create_vertex([2])
u0.add_edge(u1, [1.0])
u1.add_edge(u2, [1.0])
g2.update_weights([2.0])
print(f'g2 expectation: {g2.expectation():.6g}')
# This call should hit the cache written by fetch_compute(g) above
# because g and g2 are structurally identical and therefore share
# the same content hash.

g2 expectation: 0.5


## Pinning for mirroring

Pinning tells your local IPFS node to keep the artifact permanently. The same artifact can then be served peer-to-peer from your node to anyone else who requests the same CID — your machine becomes a mirror.

```python
phasic.pin_compute(graph)         # requires a running kubo daemon
```

Pinning is a daemon-only operation; with daemon-less consumers (HTTP gateways only) `pin_compute` raises `PTDBackendError`.

## Publishing

To contribute a new artifact, use the `phasic publish-compute` command-line tool. Given a Python file that defines a top-level `build_graph()` callable, it:

1. imports the script and runs `build_graph()` to obtain a `phasic.Graph`;
2. populates the C-side compute cache (one `expectation()` call);
3. saves the parent `.bin` and any per-SCC files to a staging directory;
4. clones `munch-group/phasic-traces`, splices a new entry into `registry.json`, copies the artifacts into `artifacts/`, and opens a pull request via the `gh` GitHub CLI.

Dry-run mode prints the entry without touching the registry:

```bash
$ cat scripts/coalescent.py
import phasic
import numpy as np

def coalescent_callback(state):
    n = state[0]
    if n <= 1:
        return []
    return [(np.array([n - 1]), [n * (n - 1) / 2])]

def build_graph():
    return phasic.Graph(
        state_length=1,
        callback=coalescent_callback,
        parameterized=True,
        nr_samples=5,
    )

$ phasic publish-compute scripts/coalescent.py \
      --id coal_n5_theta1 \
      --description "Kingman coalescent for n=5" \
      --domain population-genetics --model-type coalescent \
      --tags coalescent kingman --dry-run
{
  "coal_n5_theta1": {
    "graph_hash": "…64hex…",
    "format_revision": 2,
    "artifacts": { … },
    "metadata": { … }
  }
}
```

Without `--dry-run` the same command will clone the registry repo, make the changes, push to a feature branch on your fork, and open a PR. The maintainers then pin the artifact and merge the entry into `master`.

## Custom transport backends

If IPFS is not suitable for your deployment, subclass `TransportBackend`. A minimal S3 backend looks like this:

In [12]:
from phasic.transport import TransportBackend

class S3Backend(TransportBackend):
    """Example: read artifacts from an S3 bucket. Lazy boto3."""

    def __init__(self, bucket: str, prefix: str = 'artifacts/'):
        self.bucket = bucket
        self.prefix = prefix

    @property
    def name(self) -> str:
        return 's3'

    def _client(self):  # pragma: no cover — needs boto3
        import boto3
        return boto3.client('s3')

    def get(self, cid_or_url, output_path):
        # cid_or_url is the artifact key here, e.g. '<hash>.bin'
        print(f'S3Backend.get({cid_or_url!r}) -> {output_path}')
        # self._client().download_file(self.bucket,
        #     self.prefix + cid_or_url, str(output_path))

    def add(self, path):
        print(f'S3Backend.add({path!r})')
        # self._client().upload_file(str(path), self.bucket,
        #     self.prefix + path.name)
        return f's3://{self.bucket}/{self.prefix}{path.name}'

print('S3Backend defined (would talk to S3 with real boto3)')

S3Backend defined (would talk to S3 with real boto3)


## Summary

| Task | API |
|------|-----|
| Inspect backend  | `IPFSBackend().status()` |
| Browse registry  | `phasic.list_computes(domain=…, model_type=…, tags=…)` |
| Inspect source   | `phasic.compute_source(graph)` |
| Download artifact | `phasic.fetch_compute(graph)` |
| Pin for mirroring | `phasic.pin_compute(graph)` |
| Publish new artifact | `phasic publish-compute <script.py> --id ...` (CLI) |
| Custom transport | subclass `phasic.transport.TransportBackend` |

In [13]:
# Cleanup
shutil.rmtree(demo_root, ignore_errors=True)
if local_bin.exists():
    local_bin.unlink()
print('demo done')

demo done
